# Week 1 – Introduction to LLM access (local + cloud)
**COS30018 Intelligent Systems · Anas Al Azad**

In this notebook I call the same kind of model two ways:
* **local** – `llama3.2:1b` running on my own laptop through **Ollama**
* **cloud** – **Google Gemini** over the internet with an API key (`google-genai` library)

and then compare them on a small dataset (speed, cost, quality), try a "think step by step" prompt and put a chat window on top with Gradio.

**Before running:** the Ollama app is open and `ollama pull llama3.2:1b` has finished, and my Gemini key is in the `.env` file in the top folder of the repo (`GEMINI_API_KEY=...`). Then *Kernel → Restart & Run All* (takes about 5–10 minutes). Every number in the comparison at the end is measured by this notebook.

## Step 1 – the basics in my own words
* **LLM (large language model)** – a huge neural network trained on loads of text to guess the next word (token). You give it text (a *prompt*) and it keeps writing, which ends up being an answer.
* **API** – a set of requests one program is allowed to send another. Here my code sends the prompt as JSON and gets the answer back as JSON.
* **Jupyter Notebook** – code, notes and outputs in one document; you run it cell by cell and see the result under each cell.
* **Local vs cloud** – *local*: the model file sits on my laptop and my CPU/GPU does the work → free, private, works offline, but only small models and it's as fast as my laptop. *Cloud*: the prompt goes over the internet to Google's servers → huge fast models, but I need an account + API key, there are limits/costs and my data leaves my laptop.

In [ ]:
import json
import os
import platform
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()   # reads GEMINI_API_KEY from the .env file (looks in this folder, then the folders above)

OLLAMA_MODEL = "llama3.2:1b"
GEMINI_MODEL = "gemini-3.5-flash"     # if Google renamed/retired it, gemini_generate() below moves to the next one
GEMINI_FALLBACKS = ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-flash-lite"]
N_ROWS = 20              # same as the lab sheet (df.head(20)) - keeps me inside the free Gemini limits
RUN_ALL_LOCAL = True     # extra: local calls are free, so also run Ollama on all the reviews

Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)
results = {"run_at": time.strftime("%Y-%m-%d %H:%M"), "python": platform.python_version(),
           "os": f"{platform.system()} {platform.machine()}", "ollama_model": OLLAMA_MODEL}

print("Python", platform.python_version(), "on", platform.system(), platform.machine())
print("Gemini key found in .env:", bool(os.getenv("GEMINI_API_KEY")))

**The `time.time()` wrapper** the sheet asks for. `log_latency` goes on top of a function (`@log_latency(...)`) and writes down how long every single call took, so I can compare the two models later.

In [ ]:
LATENCY_LOG = []   # one row per model call: which model, which step, how many seconds


def log_latency(backend, step):
    def wrap(fn):
        def inner(*args, **kwargs):
            start = time.time()
            out = fn(*args, **kwargs)
            LATENCY_LOG.append({"backend": backend, "step": step, "seconds": time.time() - start})
            return out
        return inner
    return wrap

## Step 2 – Ollama + a small model (CLI)
Done in the terminal: installed Ollama from [ollama.com](https://ollama.com/download), then `ollama run llama3.2:1b` (downloads the model the first time, then you can chat; `/bye` to quit).
This cell just checks what's downloaded on my laptop.

In [ ]:
import ollama

for m in ollama.list().models:
    print(f"{m.model:16s} {m.size / 1e9:.2f} GB on disk | {m.details.parameter_size} parameters | "
          f"{m.details.quantization_level} | family: {m.details.family}")

## Step 3 – call the local model from Python (`ollama` library)
Same code as the sheet, plus a timer.

In [ ]:
start = time.time()
response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[
        {'role': 'user', 'content': 'What is an intelligent system?'}
    ]
)
ollama_hello_s = time.time() - start

print(response['message']['content'])
print(f"\n--- {OLLAMA_MODEL}: {ollama_hello_s:.1f} s | loading the model took {response.load_duration / 1e9:.1f} s "
      f"| answer = {response.eval_count} tokens ({response.eval_count / (response.eval_duration / 1e9):.0f} tokens/s) ---")

results["hello"] = {"ollama": {"seconds": round(ollama_hello_s, 2), "load_s": round(response.load_duration / 1e9, 2),
                               "answer_words": len(response['message']['content'].split()),
                               "tokens": response.eval_count,
                               "tok_per_s": round(response.eval_count / (response.eval_duration / 1e9), 1)}}

## Step 4 – what's happening under the hood
* The Ollama app runs a small **web server on my own laptop** at `http://localhost:11434` (localhost = this computer).
* `ollama.chat(...)` is just a helper – it sends an **HTTP POST request** with my prompt as JSON to `localhost:11434/api/chat`.
* The server loads the **model weights** (the ~1.3 GB file Ollama downloaded) into memory – that's why the very first call is slower – then runs the network one token at a time and sends the whole answer back as a **JSON response**.
* Nothing leaves my laptop. The cloud version (step 5) is the same idea, only the server is Google's and I need a key to prove who I am.

To prove it, here's the same kind of call **without** the `ollama` library – a plain HTTP request:

In [ ]:
import requests

payload = {"model": OLLAMA_MODEL,
           "messages": [{"role": "user", "content": "Say hello in five words."}],
           "stream": False}
r = requests.post("http://localhost:11434/api/chat", json=payload, timeout=300)

print("HTTP status:", r.status_code)
data = r.json()
print("keys in the JSON response:", list(data.keys()))
print("answer:", data["message"]["content"])
print(f"model load {data['load_duration'] / 1e9:.2f} s | read the prompt ({data['prompt_eval_count']} tokens) "
      f"{data['prompt_eval_duration'] / 1e9:.2f} s | wrote {data['eval_count']} tokens in {data['eval_duration'] / 1e9:.2f} s")

loaded = ollama.ps().models           # what's loaded in memory right now
mem_gb = loaded[0].size / 1e9 if loaded else 0
on_gpu = loaded[0].size_vram / loaded[0].size if loaded and loaded[0].size else 0
info = ollama.show(OLLAMA_MODEL)
print(f"\nloaded in memory: {mem_gb:.2f} GB ({on_gpu:.0%} of it on the GPU)")
print(f"model details: {info.details.parameter_size} parameters, {info.details.quantization_level} "
      f"(weights squeezed to 8 bits so it fits on a laptop)")

results["under_hood"] = {"http_status": r.status_code, "json_keys": list(data.keys()),
                         "mem_gb": round(mem_gb, 2), "gpu_share": round(on_gpu, 2),
                         "params": info.details.parameter_size, "quant": info.details.quantization_level,
                         "disk_gb": round(ollama.list().models[0].size / 1e9, 2)}

## Step 5 – the same prompt in the cloud (Gemini)
Made a key at [aistudio.google.com/app/api-keys](https://aistudio.google.com/app/api-keys) (*Create API key*). It goes in the `.env` file – **never** typed into the notebook, so it can't end up on GitHub or in a screenshot.

`gemini_generate()` is the sheet's `client.models.generate_content(...)` with a bit of error handling, because the free tier has limits: if Google says "too many requests" (error 429) it waits and tries again, and if a model name doesn't exist any more (404) it moves to the next model in my list. (You may see one yellow warning about "AFC / automatic function calling" the first time – that's harmless.)

In [ ]:
from google import genai
from google.genai import errors

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
GEMINI_PROBLEMS = []   # anything that went wrong (rate limits, renamed models...) - goes into my worklog
GEMINI_USAGE = []      # tokens per call


def gemini_generate(contents, tries=8):
    global GEMINI_MODEL
    for attempt in range(tries):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=contents)
            u = response.usage_metadata
            GEMINI_USAGE.append({"in": u.prompt_token_count or 0, "out": u.candidates_token_count or 0,
                                 "thinking": u.thoughts_token_count or 0})
            return response
        except errors.ClientError as e:
            msg = str(e)
            if e.code == 404 and GEMINI_FALLBACKS:                       # model name not available
                GEMINI_PROBLEMS.append(f"{GEMINI_MODEL} not found (404) -> switched to {GEMINI_FALLBACKS[0]}")
                GEMINI_MODEL = GEMINI_FALLBACKS.pop(0)
            elif e.code == 429 and "PerDay" in msg and GEMINI_FALLBACKS:  # used up today's free requests
                GEMINI_PROBLEMS.append(f"daily free limit used up on {GEMINI_MODEL} -> switched to {GEMINI_FALLBACKS[0]}")
                GEMINI_MODEL = GEMINI_FALLBACKS.pop(0)
            elif e.code == 429:                                          # too many requests per minute
                found = re.search(r"retry in ([\d.]+)s|retryDelay'?\"?:\s*'?\"?(\d+)s", msg)
                wait = float(found.group(1) or found.group(2)) + 1 if found else 15 * (attempt + 1)
                GEMINI_PROBLEMS.append(f"429 too many requests -> waited {wait:.0f} s")
                print(f"  (Gemini rate limit - waiting {wait:.0f} s)")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Gemini kept failing: {GEMINI_PROBLEMS[-3:]}")


start = time.time()
response = gemini_generate("What is an intelligent system?")
gemini_hello_s = time.time() - start

print(response.text)
u = response.usage_metadata
print(f"\n--- {GEMINI_MODEL}: {gemini_hello_s:.1f} s | prompt {u.prompt_token_count} tokens, "
      f"answer {u.candidates_token_count} tokens, hidden 'thinking' {u.thoughts_token_count or 0} tokens ---")

results["hello"]["gemini"] = {"seconds": round(gemini_hello_s, 2), "answer_words": len(response.text.split()),
                              "in_tokens": u.prompt_token_count, "out_tokens": u.candidates_token_count,
                              "thinking_tokens": u.thoughts_token_count or 0}
print(f"\nSame question: local {ollama_hello_s:.1f} s ({results['hello']['ollama']['answer_words']} words) vs "
      f"cloud {gemini_hello_s:.1f} s ({results['hello']['gemini']['answer_words']} words)")

## Step 6 – load the small dataset
`reviews.csv` from the lab materials = movie reviews with a `Sentiment` label.

In [ ]:
df = pd.read_csv("reviews.csv")
print("columns:", df.columns.tolist())

# (my fix) the column is called 'Text' (capital T), so the sheet's df['text'] gives KeyError: 'text'
df = df.rename(columns={"Text": "text", "Sentiment": "label"})
df["words"] = df["text"].str.split().str.len()

print(len(df), "reviews | labels:", df["label"].value_counts().to_dict())
print(f"review length: {df.words.min()} to {df.words.max()} words (average {df.words.mean():.0f})")
weird = df["text"].str.contains("√|Ã|â€")
print("reviews with broken characters (e.g. 'clich√©'):", weird.sum())

results["dataset"] = {"rows": len(df), "labels": {str(k): int(v) for k, v in df["label"].value_counts().items()},
                      "avg_words": round(df.words.mean()), "max_words": int(df.words.max()),
                      "broken_chars": int(weird.sum()), "used": N_ROWS}

df_all = df.copy()
df = df.head(N_ROWS).copy()   # keep it small for the lab
df.head()

## Step 7 – the same prompts on both models
The sheet's two functions with my `@log_latency` wrapper on top. `clean_label` turns answers like `Negative.` or `The sentiment is negative` into one word so I can compare them. Label 0 = negative, 1 = positive.

In [ ]:
PROMPT = "Classify sentiment as Positive, Negative, or Neutral. Reply with one word only.\n\nText: {text}"


@log_latency("ollama", "sentiment")
def get_sentiment_ollama(text):
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{'role': 'user', 'content': PROMPT.format(text=text)}]
    )
    return response['message']['content'].strip()


@log_latency("gemini", "sentiment")
def get_sentiment_gemini(text):
    response = gemini_generate(PROMPT.format(text=text))
    return (response.text or "").strip()


def clean_label(answer):
    found = re.findall(r"positive|negative|neutral", answer.lower())
    return found[0].capitalize() if found else "Other"


print(f"Ollama on {len(df)} reviews...")
df['sentiment_ollama'] = df['text'].apply(get_sentiment_ollama)
print(f"Gemini on {len(df)} reviews...")
usage_before = len(GEMINI_USAGE)
df['sentiment_gemini'] = df['text'].apply(get_sentiment_gemini)

lat = pd.DataFrame(LATENCY_LOG)
df["ollama_s"] = lat.query("backend == 'ollama' and step == 'sentiment'")["seconds"].values[-len(df):]
df["gemini_s"] = lat.query("backend == 'gemini' and step == 'sentiment'")["seconds"].values[-len(df):]
df["truth"] = df["label"].map({0: "Negative", 1: "Positive"})
for m in ["ollama", "gemini"]:
    df[m] = df["sentiment_" + m].apply(clean_label)
df[["text", "truth", "sentiment_ollama", "ollama", "ollama_s", "sentiment_gemini", "gemini", "gemini_s"]].head(10)

In [ ]:
def summarise(frame, m):
    s = frame[m + "_s"]
    return {"avg_s": round(s.mean(), 2), "median_s": round(s.median(), 2), "first_s": round(s.iloc[0], 2),
            "max_s": round(s.max(), 2), "total_s": round(s.sum(), 1),
            "accuracy": round((frame[m] == frame["truth"]).mean(), 3),
            "correct": int((frame[m] == frame["truth"]).sum()),
            "counts": frame[m].value_counts().to_dict(),
            "one_word": int((frame["sentiment_" + m].str.split().str.len() == 1).sum())}


sent = {"n": len(df), "ollama": summarise(df, "ollama"), "gemini": summarise(df, "gemini")}
used = GEMINI_USAGE[usage_before:]
sent["gemini"]["tokens_in"] = sum(x["in"] for x in used)
sent["gemini"]["tokens_out"] = sum(x["out"] for x in used)
sent["gemini"]["tokens_thinking"] = sum(x["thinking"] for x in used)

table = pd.DataFrame({m: {"avg seconds / review": sent[m]["avg_s"], "median seconds": sent[m]["median_s"],
                          "slowest call (s)": sent[m]["max_s"], "total time (s)": sent[m]["total_s"],
                          "correct (label 0 = Negative)": f"{sent[m]['correct']}/{len(df)}",
                          "answered with ONE word": f"{sent[m]['one_word']}/{len(df)}",
                          "predictions": sent[m]["counts"]} for m in ["ollama", "gemini"]})
display(table)

agree = df["ollama"] == df["gemini"]
sent["agree"] = int(agree.sum())
sent["disagree_rows"] = [{"row": int(i), "ollama": df.at[i, "ollama"], "gemini": df.at[i, "gemini"],
                          "ollama_raw": df.at[i, "sentiment_ollama"][:80], "words": int(df.at[i, "words"]),
                          "snippet": df.at[i, "text"][:160]} for i in df.index[~agree]]
print(f"The two models agreed on {agree.sum()}/{len(df)} reviews." + (" Where they disagreed:" if (~agree).any() else ""))
for d in sent["disagree_rows"]:
    print(f"\n  row {d['row']}: Ollama = {d['ollama']} ('{d['ollama_raw']}') | Gemini = {d['gemini']} | {d['words']} words")
    print(f"     \"{d['snippet']}...\"")
results["sentiment"] = sent
df.drop(columns=["text"]).to_csv("results/lab1_sentiment_results.csv")

**Extra – local is free, so run Ollama on all the reviews** (not just 20). Gemini's free tier only gives a limited number of requests per day, so I kept the cloud part to the 20 from the sheet.

In [ ]:
if RUN_ALL_LOCAL:
    start = time.time()
    df_all["ollama"] = df_all["text"].apply(get_sentiment_ollama).apply(clean_label)
    took = time.time() - start
    acc = (df_all["ollama"] == "Negative").mean()
    results["ollama_all"] = {"n": len(df_all), "seconds": round(took, 1), "accuracy": round(acc, 3),
                             "counts": df_all["ollama"].value_counts().to_dict()}
    print(f"Ollama on all {len(df_all)} reviews: {took:.0f} s, {acc:.0%} said Negative, "
          f"counts = {results['ollama_all']['counts']}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
x = range(len(df))
ax[0].plot(x, df["ollama_s"], "o-", label=f"local – {OLLAMA_MODEL}")
ax[0].plot(x, df["gemini_s"], "s-", label=f"cloud – {GEMINI_MODEL}")
ax[0].set(xlabel="review number", ylabel="seconds per call", title="Latency per review (time.time() wrapper)")
ax[0].legend()
labels = ["Negative", "Neutral", "Positive", "Other"]
w = 0.38
for k, (m, name) in enumerate([("ollama", "local"), ("gemini", "cloud")]):
    counts = [sent[m]["counts"].get(l, 0) for l in labels]
    ax[1].bar([i + (k - 0.5) * w for i in range(len(labels))], counts, w, label=name)
ax[1].set_xticks(range(len(labels)), labels)
ax[1].set(ylabel="number of reviews", title=f"What each model answered (all {len(df)} are negative)")
ax[1].legend()
plt.tight_layout()
plt.savefig("figures/1_1_latency_and_answers.png", dpi=150)
plt.show()

## Step 8 – "think step by step" vs a direct answer
Same apple question, two prompts. The small local model is random-ish (it samples words), so I ask it 3 times per prompt; Gemini once per prompt to save free requests. Right answer: 3 × 12 − 5 = **31**.

In [ ]:
QUESTION = "A store has 3 boxes of 12 apples each. 5 apples are rotten. How many good apples remain?"
PROMPTS = {
    "step by step": "Think step by step, then give your final answer.\n\nQuestion: " + QUESTION,
    "direct": "Answer with the final number only.\n\nQuestion: " + QUESTION,
}
REPEATS = {"ollama": 3, "gemini": 1}


def ask(backend, prompt):
    start = time.time()
    if backend == "ollama":
        text = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])["message"]["content"]
    else:
        text = gemini_generate(prompt).text or ""
    numbers = re.findall(r"-?\d+", text.replace(",", ""))
    final = int(numbers[-1]) if numbers else None        # the last number in the answer = its final answer
    return {"seconds": round(time.time() - start, 2), "words": len(text.split()), "final": final,
            "correct": final == 31, "answer": text.strip()}


reasoning = {"expected": 31}
for backend in ["ollama", "gemini"]:
    reasoning[backend] = {}
    for name, prompt in PROMPTS.items():
        runs = [ask(backend, prompt) for _ in range(REPEATS[backend])]
        reasoning[backend][name] = runs
        for i, run in enumerate(runs, 1):
            print(f"===== {backend} | {name} | try {i}: final = {run['final']} "
                  f"({'right' if run['correct'] else 'WRONG'}) | {run['seconds']} s | {run['words']} words")
            print(run["answer"][:600], "\n")
results["reasoning"] = reasoning

summary = pd.DataFrame({f"{b} – {p}": {"right": f"{sum(r['correct'] for r in reasoning[b][p])}/{len(reasoning[b][p])}",
                                       "avg seconds": round(sum(r['seconds'] for r in reasoning[b][p]) / len(reasoning[b][p]), 2),
                                       "avg words": round(sum(r['words'] for r in reasoning[b][p]) / len(reasoning[b][p]))}
                        for b in ["ollama", "gemini"] for p in PROMPTS})
summary

## Step 9 – chat demo with Gradio
(`GradioUI` is the smolagents wrapper we use in Week 2 – there's no agent yet, so I used Gradio's own `ChatInterface`.) A switch picks the local or the cloud model, the whole conversation is sent each time (so it remembers what I said), and every reply shows which model answered and how long it took. It opens at **http://127.0.0.1:7860** – open that in the browser for the screenshot.

In [ ]:
import gradio as gr

LOCAL, CLOUD = f"Local – Ollama {OLLAMA_MODEL}", "Cloud – Gemini"


def text_of(content):
    # Gradio gives earlier messages either as plain text or as a list of parts
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        return " ".join(text_of(c) for c in content)
    if isinstance(content, dict):
        return content.get("text") or content.get("content") or ""
    return str(content)


def chat(message, history, backend):
    msgs = [{"role": h["role"], "content": re.sub(r"\n\n\*\(.*?\)\*$", "", text_of(h["content"]))}
            for h in history if h.get("role") in ("user", "assistant")]
    msgs.append({"role": "user", "content": message})
    start = time.time()
    if backend == LOCAL:
        reply, name = ollama.chat(model=OLLAMA_MODEL, messages=msgs)["message"]["content"], OLLAMA_MODEL
    else:
        contents = [{"role": "user" if m["role"] == "user" else "model", "parts": [{"text": m["content"]}]} for m in msgs]
        reply, name = gemini_generate(contents).text, GEMINI_MODEL
    return f"{reply}\n\n*({name}, {time.time() - start:.1f} s)*"


demo = gr.ChatInterface(
    chat,
    additional_inputs=[gr.Radio([LOCAL, CLOUD], value=LOCAL, label="Which model answers?")],
    additional_inputs_accordion=gr.Accordion("Model (local or cloud)", open=True),
    title="Week 1 – local vs cloud LLM chat",
    description="Anas Al Azad · COS30018 · pick a model below, then ask anything",
    examples=[["What is an intelligent system?", LOCAL], ["Explain an API to a 10 year old.", CLOUD]],
)
demo.launch(prevent_thread_lock=True)
results["gradio"] = True

## Deliverable – comparison table
Built from the numbers measured above (so it always matches this run).

In [ ]:
o, g = results["sentiment"]["ollama"], results["sentiment"]["gemini"]
h = results["hello"]
rs = results["reasoning"]


def right(b, p):
    runs = rs[b][p]
    return f"{sum(r['correct'] for r in runs)}/{len(runs)}"


rows = [
    ("Setup", f"install the Ollama app + download the model ({results['under_hood']['disk_gb']} GB), no account or key",
     "make a Google account + API key, keep the key secret (.env), install `google-genai`"),
    ("Latency – 'What is an intelligent system?'", f"{h['ollama']['seconds']} s ({h['ollama']['answer_words']} words)",
     f"{h['gemini']['seconds']} s ({h['gemini']['answer_words']} words)"),
    (f"Latency – sentiment, {results['sentiment']['n']} reviews", f"avg {o['avg_s']} s/review, total {o['total_s']} s",
     f"avg {g['avg_s']} s/review, total {g['total_s']} s"),
    ("Cost", f"$0 – runs on my laptop (uses ~{results['under_hood']['mem_gb']} GB of RAM while loaded, my battery/CPU)",
     f"$0 on the free tier (limited requests per day); used {g['tokens_in'] + g['tokens_out'] + g['tokens_thinking']:,} tokens "
     f"for the sentiment run ({g['tokens_thinking']:,} of them hidden 'thinking') – on a paid plan you pay per token"),
    ("Quality – sentiment (all 20 are negative)", f"{o['correct']}/{results['sentiment']['n']} right, "
     f"{o['one_word']}/{results['sentiment']['n']} one-word answers", f"{g['correct']}/{results['sentiment']['n']} right, "
     f"{g['one_word']}/{results['sentiment']['n']} one-word answers"),
    ("Quality – apples question (31)", f"step by step {right('ollama', 'step by step')}, direct {right('ollama', 'direct')}",
     f"step by step {right('gemini', 'step by step')}, direct {right('gemini', 'direct')}"),
    ("Privacy / offline", "nothing leaves my laptop, works without internet", "reviews are sent to Google, needs internet"),
]
md_table = "| | Local – Ollama " + OLLAMA_MODEL + " | Cloud – Gemini " + GEMINI_MODEL + " |\n|---|---|---|\n"
md_table += "\n".join(f"| **{a}** | {b} | {c} |" for a, b, c in rows)
results["comparison_rows"] = rows
display(Markdown(md_table))

## Checkpoint questions
**1. What's the practical difference in setup complexity between local and cloud?**
Local: install one app and download the model once, then it just works – no account, no key, no limits, works offline. The catch is it uses my laptop's memory and is only as fast as my laptop, and I can only run small models. Cloud: no download, a much bigger model, but I needed a Google account, an API key that I have to keep secret (`.env`, never in the code), and I have to live with the free-tier limits (error 429 when I go too fast).

**2. Where did the two models disagree, and why might that happen?** – printed below from this run.

In [ ]:
s = results["sentiment"]
lines = [f"They agreed on **{s['agree']}/{s['n']}** reviews."]
if s["disagree_rows"]:
    lines.append("They disagreed on " + ", ".join(f"row {d['row']} (Ollama: {d['ollama']}, Gemini: {d['gemini']})"
                                                for d in s["disagree_rows"]) + ".")
    lines.append("Why it might happen:")
else:
    lines.append("So no disagreements this time, but they easily could, because:")
lines.append(f"the local model only has {results['under_hood']['params']} parameters (Google doesn't say how big Gemini "
             "is, but it's far bigger), so long or mixed reviews (praising some parts but hating the film overall), "
             "sarcasm and comparisons to other films are harder for it. It also picks words with some randomness, so the "
             "same review can get a different answer next time, and it doesn't always stick to 'one word only'. "
             "And the prompt allows 'Neutral' even though the dataset only has 0/1 labels, so 'Neutral' counts as wrong.")
results["checkpoint_q2"] = " ".join(lines)
display(Markdown(results["checkpoint_q2"]))

results["gemini_model_used"] = GEMINI_MODEL
results["gemini_problems"] = GEMINI_PROBLEMS
results["latency_log"] = LATENCY_LOG
with open("results/lab1_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("saved results/lab1_results.json (my worklog builder reads this) | Gemini problems:", GEMINI_PROBLEMS or "none")